In [1]:
import pandas as pd

In [2]:
import pandas as pd

season = "2627"
praefix = season[:2]  
leagues = {
    "pl": "E0",  
    "de": "D1",  
    "es": "SP1",  
    "it": "I1",  
    "fr": "F1", 
}
urls = {
    name: f"https://football-data.co.uk/mmz4281/{season}/{code}.csv"
    for name, code in leagues.items()
}

In [3]:
import numpy as np


In [4]:
def snake_case(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


In [5]:
column_mapping = {
    "hometeam": "home_team",
    "awayteam": "away_team",
    "fthg": "full_time_home_goals",
    "ftag": "full_time_away_goals",
    "ftr": "full_time_result",
    "hthg": "half_time_home_goals",
    "htag": "half_time_away_goals",
    "htr": "half_time_result",
    "hs": "home_shots",
    "as": "away_shots",
    "hst": "home_shots_on_target",
    "ast": "away_shots_on_target"
}



In [6]:
columns_for_standings = [
    "date",
    "home_team",
    "away_team",
    "full_time_home_goals",
    "full_time_away_goals",
    "full_time_result",
]

In [7]:
league_data = {}

for name, code in leagues.items():
    league_data[name] = {}

    full = pd.read_csv(urls[name])
    full = snake_case(full)
    full = full.rename(columns=column_mapping)

    league_data[name]["full"] = full
    league_data[name]["matches"] = full[columns_for_standings].copy()

    matches = league_data[name]["matches"]

    # Combine home and away teams, remove duplicates, then sort them.
    league_data[name]["teams"] = sorted(
        set(matches["home_team"]) | set(matches["away_team"])
    )

    league_data[name]["team_matches"] = {}

    for team in league_data[name]["teams"]:
        team_matches = matches[
            (matches["home_team"] == team)
            | (matches["away_team"] == team)
        ].copy()

        league_data[name]["team_matches"][team] = team_matches


In [8]:
for name in leagues:
    standings_rows = []

    for team in league_data[name]["teams"]:
        club_df = league_data[name]["team_matches"][team]

        
        played = club_df.dropna(
            subset=["full_time_home_goals", "full_time_away_goals"]
        )

        home_games = played[played["home_team"] == team]
        away_games = played[played["away_team"] == team]

        home_wins = (
            home_games["full_time_home_goals"]
            > home_games["full_time_away_goals"]
        ).sum()
        away_wins = (
            away_games["full_time_away_goals"]
            > away_games["full_time_home_goals"]
        ).sum()
        wins = home_wins + away_wins

        home_draws = (
            home_games["full_time_home_goals"]
            == home_games["full_time_away_goals"]
        ).sum()
        away_draws = (
            away_games["full_time_away_goals"]
            == away_games["full_time_home_goals"]
        ).sum()
        draws = home_draws + away_draws

        home_losses = (
            home_games["full_time_home_goals"]
            < home_games["full_time_away_goals"]
        ).sum()
        away_losses = (
            away_games["full_time_away_goals"]
            < away_games["full_time_home_goals"]
        ).sum()
        losses = home_losses + away_losses

        goals_for = (
            home_games["full_time_home_goals"].sum()
            + away_games["full_time_away_goals"].sum()
        )
        goals_against = (
            home_games["full_time_away_goals"].sum()
            + away_games["full_time_home_goals"].sum()
        )
        goal_difference = goals_for - goals_against
        points = 3 * wins + draws

        standings_rows.append({
            "Team": team,
            "Played": len(played),
            "W": wins,
            "D": draws,
            "L": losses,
            "GF": goals_for,
            "GA": goals_against,
            "GD": goal_difference,
            "Points": points,
        })

    league_data[name]["standings"] = (
        pd.DataFrame(standings_rows)
        .sort_values(
            ["Points", "GD", "GF"],
            ascending=False,
        )
        .reset_index(drop=True)
    )


In [9]:
for name in leagues:
    standings_rows = []

    for team in league_data[name]["teams"]:
        club_df = league_data[name]["team_matches"][team]

        
        played = club_df.dropna(
            subset=["full_time_home_goals", "full_time_away_goals"]
        )

        home_games = played[played["home_team"] == team]
        away_games = played[played["away_team"] == team]

        home_wins = (
            home_games["full_time_home_goals"]
            > home_games["full_time_away_goals"]
        ).sum()
        away_wins = (
            away_games["full_time_away_goals"]
            > away_games["full_time_home_goals"]
        ).sum()
        wins = home_wins + away_wins

        home_draws = (
            home_games["full_time_home_goals"]
            == home_games["full_time_away_goals"]
        ).sum()
        away_draws = (
            away_games["full_time_away_goals"]
            == away_games["full_time_home_goals"]
        ).sum()
        draws = home_draws + away_draws

        home_losses = (
            home_games["full_time_home_goals"]
            < home_games["full_time_away_goals"]
        ).sum()
        away_losses = (
            away_games["full_time_away_goals"]
            < away_games["full_time_home_goals"]
        ).sum()
        losses = home_losses + away_losses

        goals_for = (
            home_games["full_time_home_goals"].sum()
            + away_games["full_time_away_goals"].sum()
        )
        goals_against = (
            home_games["full_time_away_goals"].sum()
            + away_games["full_time_home_goals"].sum()
        )
        goal_difference = goals_for - goals_against
        points = 3 * wins + draws

        standings_rows.append({
            "Team": team,
            "Played": len(played),
            "W": wins,
            "D": draws,
            "L": losses,
            "GF": goals_for,
            "GA": goals_against,
            "GD": goal_difference,
            "Points": points,
        })

    league_data[name]["standings"] = (
        pd.DataFrame(standings_rows)
        .sort_values(
            ["Points", "GD", "GF"],
            ascending=False,
        )
        .reset_index(drop=True)
    )


In [10]:
standings_dfs = {}
for name in leagues:
    standings_dfs[name] = league_data[name]["standings"]


In [11]:
full_dfs = {}

for name in leagues:
    full_dfs[name] = league_data[name]["full"]

In [17]:
full_dfs["pl"]
step = 10
for start in range(10, len(full_dfs["pl"].columns), step):
    blok = full_dfs["pl"].columns[start : start + step]
    print(f"\n--- Kolone {start} do {start + len(blok) - 1} ---")
    print(full_dfs["pl"][blok].head(3))


--- Kolone 10 do 19 ---
  half_time_result    referee  home_xg  away_xg  home_shots  away_shots  \
0                H  T Bramall     1.88     0.20          20           4   
1                H  D England     1.01     1.83           8          21   
2                H  P Tierney     1.12     1.96          16          11   

   home_shots_on_target  away_shots_on_target  home_fouls  away_fouls  
0                     6                     1          10          13  
1                     4                     5          10           9  
2                     5                     4           5          15  

--- Kolone 20 do 29 ---
   home_corners  away_corners  home_yellow_cards  away_yellow_cards  \
0             8             2                  1                  1   
1             1             6                  2                  1   
2             4             0                  0                  1   

   home_red_cards  away_red_cards  b365_home_odds  b365_draw_odds  \
0      

In [13]:
column_mapping = {
    # Statističke i meč kolone (iz tvog prvog bloka)
    "hxg": "home_xg",  # Expected Goals (xG) domaćina
    "axg": "away_xg",  # Expected Goals (xG) gosta
    "home_shots": "home_shots",  # Ukupni šutevi domaćina (u sirovom fajlu često 'HS')
    "away_shots": "away_shots",  # Ukupni šutevi gosta (u sirovom fajlu često 'AS')
    "home_shots_on_target": "home_shots_on_target",  # Šutevi u okvir gola domaćina (često 'HST')
    "away_shots_on_target": "away_shots_on_target",  # Šutevi u okvir gola gosta (često 'AST')
    "hf": "home_fouls",  # Faulovi domaćina
    "af": "away_fouls",  # Faulovi gosta
    # Kolone 20 do 29
    "hc": "home_corners",  # Korneri domaćina
    "ac": "away_corners",  # Korneri gosta
    "hy": "home_yellow_cards",  # Žuti kartoni domaćina
    "ay": "away_yellow_cards",  # Žuti kartoni gosta
    "hr": "home_red_cards",  # Crveni kartoni domaćina
    "ar": "away_red_cards",  # Crveni kartoni gosta
    "b365h": "b365_home_odds",  # Bet365 kvota – pobjeda domaćina (Home)
    "b365d": "b365_draw_odds",  # Bet365 kvota – neriješeno (Draw)
    "b365a": "b365_away_odds",  # Bet365 kvota – pobjeda gosta (Away)
    "bfdh": "bfd_home_odds",  # Betfair kvota – pobjeda domaćina
    # Kolone 30 do 39
    "bfdd": "bfd_draw_odds",  # Betfair kvota – neriješeno
    "bfda": "bfd_away_odds",  # Betfair kvota – pobjeda gosta
    "bvh": "bv_home_odds",  # BetVictor kvota – pobjeda domaćina
    "bvd": "bv_draw_odds",  # BetVictor kvota – neriješeno
    "bva": "bv_away_odds",  # BetVictor kvota – pobjeda gosta
    "bwh": "bw_home_odds",  # Bwin kvota – pobjeda domaćina
    "bwd": "bw_draw_odds",  # Bwin kvota – neriješeno
    "bwa": "bw_away_odds",  # Bwin kvota – pobjeda gosta
    "pph": "pp_home_odds",  # Paddy Power kvota – pobjeda domaćina
    "ppd": "pp_draw_odds",  # Paddy Power kvota – neriješeno
    # Kolone 40 do 49
    "ppa": "pp_away_odds",  # Paddy Power – pobjeda gosta
    "skbh": "skb_home_odds",  # Sky Bet – pobjeda domaćina
    "skbd": "skb_draw_odds",  # Sky Bet – neriješeno
    "skba": "skb_away_odds",  # Sky Bet – pobjeda gosta
    "maxh": "max_home_odds",  # Najveća tržišna kvota – domaćin
    "maxd": "max_draw_odds",  # Najveća tržišna kvota – neriješeno
    "maxa": "max_away_odds",  # Najveća tržišna kvota – gost
    "avgh": "avg_home_odds",  # Prosječna tržišna kvota – domaćin
    "avgd": "avg_draw_odds",  # Prosječna tržišna kvota – neriješeno
    "avga": "avg_away_odds",  # Prosječna tržišna kvota – gost
    # Kolone 50 do 59
    "bfeh": "bfe_home_odds",  # Betfair Exchange – domaćin
    "bfed": "bfe_draw_odds",  # Betfair Exchange – neriješeno
    "bfea": "bfe_away_odds",  # Betfair Exchange – gost
    "b365>2.5": "b365_over_2_5_odds",  # Bet365 – više od 2.5 gola
    "b365<2.5": "b365_under_2_5_odds",  # Bet365 – manje od 2.5 gola
    "max>2.5": "max_over_2_5_odds",  # Najveća kvota – više od 2.5 gola
    "max<2.5": "max_under_2_5_odds",  # Najveća kvota – manje od 2.5 gola
    "avg>2.5": "avg_over_2_5_odds",  # Prosječna kvota – više od 2.5 gola
    "avg<2.5": "avg_under_2_5_odds",  # Prosječna kvota – manje od 2.5 gola
    "bfe>2.5": "bfe_over_2_5_odds",  # Betfair Exchange – više od 2.5 gola
    # Kolone 60 do 69
    "bfe<2.5": "bfe_under_2_5_odds",  # Betfair Exchange – manje od 2.5 gola
    "ahh": "asian_handicap_line",  # Linija azijskog hendikepa (Home)
    "b365ahh": "b365_ah_home_odds",  # Bet365 Asian Handicap – domaćin
    "b365aha": "b365_ah_away_odds",  # Bet365 Asian Handicap – gost
    "maxahh": "max_ah_home_odds",  # Najveća kvota Asian Handicap – domaćin
    "maxaha": "max_ah_away_odds",  # Najveća kvota Asian Handicap – gost
    "avgahh": "avg_ah_home_odds",  # Prosječna kvota Asian Handicap – domaćin
    "avgaha": "avg_ah_away_odds",  # Prosječna kvota Asian Handicap – gost
    "bfeahh": "bfe_ah_home_odds",  # Betfair Exchange Asian Handicap – domaćin
    "bfeaha": "bfe_ah_away_odds",  # Betfair Exchange Asian Handicap – gost
    # Kolone 70 do 79
    "b365ch": "b365_close_home_odds",  # Bet365 closing kvota – domaćin
    "b365cd": "b365_close_draw_odds",  # Bet365 closing kvota – neriješeno
    "b365ca": "b365_close_away_odds",  # Bet365 closing kvota – gost
    "bfdch": "bfd_close_home_odds",  # Betfair closing kvota – domaćin
    "bfdcd": "bfd_close_draw_odds",  # Betfair closing kvota – neriješeno
    "bfdca": "bfd_close_away_odds",  # Betfair closing kvota – gost
    "bvch": "bv_close_home_odds",  # BetVictor closing kvota – domaćin
    "bvcd": "bv_close_draw_odds",  # BetVictor closing kvota – neriješeno
    "bvca": "bv_close_away_odds",  # BetVictor closing kvota – gost
    "bwch": "bw_close_home_odds",  # Bwin closing kvota – domaćin
    # Kolone 80 do 89
    "bwcd": "bw_close_draw_odds",  # Bwin closing – neriješeno
    "bwca": "bw_close_away_odds",  # Bwin closing – pobjeda gosta
    "ppch": "pp_close_home_odds",  # Paddy Power closing – pobjeda domaćina
    "ppcd": "pp_close_draw_odds",  # Paddy Power closing – neriješeno
    "ppca": "pp_close_away_odds",  # Paddy Power closing – pobjeda gosta
    "skbch": "skb_close_home_odds",  # Sky Bet closing – pobjeda domaćina
    "skbcd": "skb_close_draw_odds",  # Sky Bet closing – neriješeno
    "skbca": "skb_close_away_odds",  # Sky Bet closing – pobjeda gosta
    "maxch": "max_close_home_odds",  # Najveća closing kvota – domaćin
    "maxcd": "max_close_draw_odds",  # Najveća closing kvota – neriješeno
    # Kolone 90 do 99
    "maxca": "max_close_away_odds",  # Najveća closing kvota – gost
    "avgch": "avg_close_home_odds",  # Prosječna closing kvota – domaćin
    "avgcd": "avg_close_draw_odds",  # Prosječna closing kvota – neriješeno
    "avgca": "avg_close_away_odds",  # Prosječna closing kvota – gost
    "bfech": "bfe_close_home_odds",  # Betfair Exchange closing – domaćin
    "bfecd": "bfe_close_draw_odds",  # Betfair Exchange closing – neriješeno
    "bfeca": "bfe_close_away_odds",  # Betfair Exchange closing – gost
    "b365c>2.5": "b365_close_over_2_5_odds",  # Bet365 closing – više od 2.5 gola
    "b365c<2.5": "b365_close_under_2_5_odds",  # Bet365 closing – manje od 2.5 gola
    "maxc>2.5": "max_close_over_2_5_odds",  # Najveća closing kvota – više od 2.5 gola
    # Kolone 100 do 109
    "maxc<2.5": "max_close_under_2_5_odds",  # Najveća closing kvota – manje od 2.5 gola
    "avgc>2.5": "avg_close_over_2_5_odds",  # Prosječna closing kvota – više od 2.5 gola
    "avgc<2.5": "avg_close_under_2_5_odds",  # Prosječna closing kvota – manje od 2.5 gola
    "bfec>2.5": "bfe_close_over_2_5_odds",  # Betfair Exchange closing – više od 2.5 gola
    "bfec<2.5": "bfe_close_under_2_5_odds",  # Betfair Exchange closing – manje od 2.5 gola
    "ahch": "asian_handicap_close_line",  # Zaključna linija azijskog hendikepa (Home)
    "b365cahh": "b365_close_ah_home_odds",  # Bet365 closing Asian Handicap – domaćin
    "b365caha": "b365_close_ah_away_odds",  # Betfair/Bet365 closing Asian Handicap – gost
    "maxcahh": "max_close_ah_home_odds",  # Najveća closing kvota AH – domaćin
    "maxcaha": "max_close_ah_away_odds",  # Najveća closing kvota AH – gost
    # Kolone 110 do 113
    "avgcahh": "avg_close_ah_home_odds",  # Prosječna closing kvota AH – domaćin
    "avgcaha": "avg_close_ah_away_odds",  # Prosječna closing kvota AH – gost
    "bfecahh": "bfe_close_ah_home_odds",  # Betfair Exchange closing AH – domaćin
    "bfecaha": "bfe_close_ah_away_odds",  # Betfair Exchange closing AH – gost
}


In [14]:
for name in leagues.keys():
    full_dfs[name] = full_dfs[name].rename(columns=column_mapping)


In [15]:
full_dfs["de"]

,div,date,time,home_team,away_team,full_time_home_goals,full_time_away_goals,full_time_result,half_time_home_goals,half_time_away_goals,...,bfe_close_under_2_5_odds,asian_handicap_close_line,b365_close_ah_home_odds,b365_close_ah_away_odds,max_close_ah_home_odds,max_close_ah_away_odds,avg_close_ah_home_odds,avg_close_ah_away_odds,bfe_close_ah_home_odds,bfe_close_ah_away_odds
0,D1,28/08/2026,19:30,Bayern Munich,Stuttgart,5,1,H,1,0,...,5.70,-2.00,1.83,2.03,1.83,2.03,1.77,1.98,1.90,2.10
1,D1,29/08/2026,14:30,Elversberg,Leverkusen,3,2,H,2,0,...,2.78,1.00,1.90,1.95,1.90,1.95,1.84,1.90,1.94,2.02
2,D1,29/08/2026,14:30,FC Koln,Hoffenheim,3,2,H,0,1,...,2.54,0.25,1.95,1.90,1.95,1.90,1.88,1.86,2.03,1.94
3,D1,29/08/2026,14:30,Mainz,Paderborn,0,0,D,0,0,...,2.86,-1.00,1.83,2.03,1.83,2.03,1.79,1.96,1.92,2.05
4,D1,29/08/2026,14:30,RB Leipzig,M'gladbach,3,0,H,1,0,...,2.96,-1.00,1.88,1.98,1.88,1.98,1.84,1.91,1.98,2.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,D1,21/11/2026,14:30,M'gladbach,Dortmund,0,1,A,0,0,...,2.74,0.75,1.88,1.98,1.93,1.98,1.87,1.87,2.03,1.96
86,D1,21/11/2026,14:30,Union Berlin,RB Leipzig,1,5,A,1,1,...,2.98,1.25,1.90,1.95,1.90,1.95,1.82,1.91,1.92,2.04
87,D1,21/11/2026,17:30,Schalke 04,Stuttgart,0,0,D,0,0,...,3.00,0.50,1.85,2.00,1.95,2.00,1.83,1.90,1.92,2.06
88,D1,22/11/2026,14:30,Augsburg,Mainz,3,1,H,2,0,...,3.15,0.25,1.93,1.93,1.93,1.93,1.86,1.88,1.96,2.02


In [18]:
file_path = "football_standings.xlsx"

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
    for name in leagues:
        table = league_data[name]["standings"].copy()
        table.index = range(1, len(table) + 1)

        table.to_excel(
            writer,
            sheet_name=name.upper(),
            index_label="Pos"
        )